# RW Aur A–B Global Parameter Search

Server notebook for a coarse simultaneous search over:
- pericentre radius `Rp`
- initial disc radius `Rout`
- eccentricity `e`
- pericentre direction
- disc `theta`
- disc `phi`
- time after pericentre

Every snapshot is projected to the observed disc orientation before the A–B likelihood is evaluated.


In [1]:
import numpy as np
import rebound
import csv
import os
import time

M_A = 1.4
M_B = 0.9
R_IN = 5.9
START_SEP_FACTOR = 10.0
AU_PER_YR_TO_KMS = 4.74047

OBSERVED_AB_SEP_AU = 210.0
OBSERVED_AB_PA_DEG = 254.14
OBSERVED_DISC_INC_DEG = 55.51
OBSERVED_DISC_PA_DEG = 41.05

SIGMA_SEP_AU = 20.0
SIGMA_PA_DEG = 5.0


In [2]:
def _omega_for_tilt(theta, phi):
    return theta, np.pi/2 + phi

def add_binary(sim, m_a=M_A, m_b=M_B, r_peri=None, ecc=None,
               start_sep_factor=START_SEP_FACTOR, peri_direction_deg=None):
    sim.add(m=m_a, name="A")
    starA = sim.particles[0]
    a = r_peri/(1-ecc)
    p = r_peri*(1+ecc)
    r_start = start_sep_factor*r_peri
    cos_f = np.clip((p/r_start - 1)/ecc, -1.0, 1.0)
    f_start = -np.arccos(cos_f)
    sim.add(m=m_b, a=a, e=ecc, inc=0.0, Omega=0.0, omega=np.radians(peri_direction_deg),
            f=f_start, primary=starA, name="B")
    sim.move_to_com()
    return sim.particles[0], sim.particles[1]

def add_disc(sim, primary, n_particles, r_in=R_IN, r_out=None,
             theta_disc=None, phi_disc=None, seed=None):
    rng = np.random.default_rng(seed)
    inc, Omega = _omega_for_tilt(theta_disc, phi_disc)
    R = rng.uniform(r_in, r_out, n_particles)
    f = rng.uniform(0, 2*np.pi, n_particles)
    primary_index = primary.index
    for Ri, fi in zip(R, f):
        sim.add(m=0.0, a=Ri, e=0.0, inc=inc, Omega=Omega, omega=0.0, f=fi,
                primary=sim.particles[primary_index])
    return sim, R, f

def build_simulation(n_disc, seed, r_out, theta_disc, phi_disc,
                     r_peri, ecc, peri_direction_deg, integrator="ias15", r_in=R_IN):
    sim = rebound.Simulation()
    sim.G = 4*np.pi**2
    sim.integrator = integrator
    starA, _ = add_binary(sim, r_peri=r_peri, ecc=ecc, peri_direction_deg=peri_direction_deg)
    _, R0, f0 = add_disc(sim, starA, n_disc, r_in=r_in, r_out=r_out,
                         theta_disc=theta_disc, phi_disc=phi_disc, seed=seed)
    sim.N_active = 2
    return sim, R0, f0


In [3]:
def find_pericentre_time(r_peri, ecc, peri_direction_deg=200.0):
    sim = rebound.Simulation()
    sim.G = 4*np.pi**2
    sim.integrator = "ias15"
    add_binary(sim, r_peri=r_peri, ecc=ecc, peri_direction_deg=peri_direction_deg)

    times = np.linspace(0, 5000, 5001)
    separations = []
    for t in times:
        sim.integrate(t)
        A, B = sim.particles[0], sim.particles[1]
        separations.append(np.sqrt((B.x-A.x)**2 + (B.y-A.y)**2 + (B.z-A.z)**2))
    return times[np.argmin(separations)]


In [4]:
def _observer_basis(theta_obs, phi_obs, pa_rotation=0.0):
    theta_obs = np.pi - theta_obs
    los = np.array([np.sin(theta_obs)*np.cos(phi_obs), np.sin(theta_obs)*np.sin(phi_obs), np.cos(theta_obs)])
    e_theta = np.array([np.cos(theta_obs)*np.cos(phi_obs), np.cos(theta_obs)*np.sin(phi_obs), -np.sin(theta_obs)])
    e_phi = np.array([-np.sin(phi_obs), np.cos(phi_obs), 0.0])
    c, s = np.cos(pa_rotation), np.sin(pa_rotation)
    north = c*e_theta + s*e_phi
    east = c*e_phi - s*e_theta
    return east, north, los

def project_positions(rel_pos, theta_obs, phi_obs, pa_rotation=0.0):
    east, north, los = _observer_basis(theta_obs, phi_obs, pa_rotation)
    return rel_pos@east, rel_pos@north, rel_pos@los

def calibrate_pa_rotation(sim, theta_obs, phi_obs, target_AB_PA_deg, n_scan=720):
    A, B = sim.particles[0], sim.particles[1]
    rel = np.array([[B.x-A.x, B.y-A.y, B.z-A.z]])
    best_pa, best_diff = 0.0, np.inf

    for pa_deg in np.linspace(0, 360, n_scan, endpoint=False):
        east, north, _ = _observer_basis(theta_obs, phi_obs, np.radians(pa_deg))
        e, n = rel[0]@east, rel[0]@north
        pa_model = np.degrees(np.arctan2(e, n)) % 360
        diff = min(abs(pa_model-target_AB_PA_deg), 360-abs(pa_model-target_AB_PA_deg))
        if diff < best_diff:
            best_diff, best_pa = diff, pa_deg

    east, north, _ = _observer_basis(theta_obs, phi_obs, np.radians(best_pa))
    e, n = rel[0]@east, rel[0]@north
    return best_pa, np.degrees(np.arctan2(e, n)) % 360, np.hypot(e, n)


In [5]:
def rotate_final_frame_to_observation(sim_final, target_inc_deg=55.51, target_pa_deg=41.05,
                                      disc_radius=30.0, theta_obs_deg=76.0, phi_obs_deg=114.0,
                                      target_ab_pa_deg=254.14):
    A, B = sim_final.particles[0], sim_final.particles[1]

    theta_obs, phi_obs = np.radians(theta_obs_deg), np.radians(phi_obs_deg)
    pa_deg, _, _ = calibrate_pa_rotation(sim_final, theta_obs, phi_obs, target_ab_pa_deg)
    pa_rad = np.radians(pa_deg)

    pos = np.array([[p.x-A.x, p.y-A.y, p.z-A.z] for p in sim_final.particles])
    vel = np.array([[p.vx-A.vx, p.vy-A.vy, p.vz-A.vz] for p in sim_final.particles])

    east, north, los = project_positions(pos, theta_obs, phi_obs, pa_rad)
    veast, vnorth, vlos = project_positions(vel, theta_obs, phi_obs, pa_rad)
    pos_sky = np.column_stack((east, north, los))
    vel_sky = np.column_stack((veast, vnorth, vlos))

    tracers = np.arange(2, len(pos_sky))
    disc_indices = tracers[np.linalg.norm(pos[tracers], axis=1) < disc_radius]
    if len(disc_indices) < 5:
        raise ValueError(f"Only {len(disc_indices)} tracers within {disc_radius} AU of A.")

    L = np.cross(pos_sky[disc_indices], vel_sky[disc_indices])
    Ln = np.linalg.norm(L, axis=1)
    Lhat = np.mean(L[Ln > 0] / Ln[Ln > 0, None], axis=0)
    Lhat /= np.linalg.norm(Lhat)
    if Lhat[2] < 0: Lhat = -Lhat

    inc, pa = np.radians(target_inc_deg), np.radians(target_pa_deg)
    target1 = np.array([np.sin(inc)*np.cos(pa), -np.sin(inc)*np.sin(pa), np.cos(inc)])
    target2 = np.array([-np.sin(inc)*np.cos(pa), np.sin(inc)*np.sin(pa), np.cos(inc)])
    angle1 = np.arccos(np.clip(np.dot(Lhat, target1), -1, 1))
    angle2 = np.arccos(np.clip(np.dot(Lhat, target2), -1, 1))
    target_L, angle = (target1, angle1) if angle1 <= angle2 else (target2, angle2)

    axis = np.cross(Lhat, target_L)
    axis_norm = np.linalg.norm(axis)
    if axis_norm < 1e-12:
        R = np.eye(3)
    else:
        axis /= axis_norm
        x, y, z = axis
        K = np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])
        R = np.eye(3) + np.sin(angle)*K + (1-np.cos(angle))*(K @ K)

    pos_rot = (R @ pos_sky.T).T
    vel_rot = (R @ vel_sky.T).T

    Lnew = np.cross(pos_rot[disc_indices], vel_rot[disc_indices])
    Ln = np.linalg.norm(Lnew, axis=1)
    Lnew = np.mean(Lnew[Ln > 0] / Ln[Ln > 0, None], axis=0)
    Lnew /= np.linalg.norm(Lnew)
    if Lnew[2] < 0: Lnew = -Lnew

    final_inc = np.degrees(np.arccos(np.clip(Lnew[2], -1, 1)))
    major = np.cross([0, 0, 1], Lnew)
    final_pa = np.degrees(np.arctan2(major[0], major[1])) % 180

    info = {"inclination_after": final_inc, "pa_after": final_pa,
            "rotation_deg": np.degrees(angle), "disc_indices": disc_indices}
    return pos_rot, vel_rot, info


In [6]:
def angle_difference(a, b):
    return (a - b + 180) % 360 - 180

def log_likelihood_AB(sep_AU, ab_pa_deg, obs_sep=OBSERVED_AB_SEP_AU,
                      obs_pa=OBSERVED_AB_PA_DEG, sigma_sep=SIGMA_SEP_AU,
                      sigma_pa=SIGMA_PA_DEG):
    dsep = (sep_AU - obs_sep) / sigma_sep
    dpa = angle_difference(ab_pa_deg, obs_pa) / sigma_pa
    return -0.5 * (dsep**2 + dpa**2)


## Global parameter ranges

Start coarse. Time is evaluated as sequential snapshots, so it does not require a separate integration for each value.

Edit only this cell for the server search.


In [7]:
R_PERI_VALUES = [55.0, 65.0, 75.0]
R_OUT_VALUES = [60.0, 70.0, 80.0]
ECC_VALUES = [0.90, 0.95, 1.01]
PERI_DIR_VALUES = [180.0, 200.0, 220.0]
THETA_VALUES = [10.0, 15.0, 20.0]
PHI_VALUES = [330.0, 350.0, 370.0]
TIME_VALUES = [400.0, 500.0, 600]

N_DISC = 300
SEED = 0
OUTPUT_CSV = "rw_aur_global_search.csv"

n_integrations = (len(R_PERI_VALUES)*len(R_OUT_VALUES)*len(ECC_VALUES)*
                  len(PERI_DIR_VALUES)*len(THETA_VALUES)*len(PHI_VALUES))
n_scores = n_integrations * len(TIME_VALUES)

print(f"Physical integrations: {n_integrations}")
print(f"Scored snapshots:      {n_scores}")


Physical integrations: 729
Scored snapshots:      2187


In [8]:
def global_parameter_search(r_peri_values, r_out_values, ecc_values, peri_dir_values,
                            theta_values, phi_values, time_values, n_disc=300, seed=0,
                            output_csv="rw_aur_global_search.csv"):
    fields = ["r_peri","r_out","ecc","peri_direction_deg","theta_disc_deg","phi_disc_deg",
              "t_after_peri","sep_AU","ab_pa_deg","inc_after","pa_after","logL"]
    write_header = not os.path.exists(output_csv)
    total = (len(r_peri_values)*len(r_out_values)*len(ecc_values)*len(peri_dir_values)*
             len(theta_values)*len(phi_values))
    count = 0
    start = time.time()
    peri_cache = {}

    with open(output_csv, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        if write_header: writer.writeheader()

        for r_peri in r_peri_values:
            for ecc in ecc_values:
                for peri_dir in peri_dir_values:
                    key = (r_peri, ecc, peri_dir)
                    if key not in peri_cache:
                        peri_cache[key] = find_pericentre_time(r_peri, ecc, peri_dir)
                    t_peri = peri_cache[key]

                    for r_out in r_out_values:
                        for theta in theta_values:
                            for phi in phi_values:
                                count += 1
                                sim, _, _ = build_simulation(
                                    n_disc, seed, r_out, np.radians(theta), np.radians(phi),
                                    r_peri, ecc, peri_dir)

                                for t_after in sorted(time_values):
                                    sim.integrate(t_peri + t_after)
                                    pos_rot, _, rot = rotate_final_frame_to_observation(
                                        sim, OBSERVED_DISC_INC_DEG, OBSERVED_DISC_PA_DEG, 30.0)

                                    east_B, north_B = pos_rot[1, 0], pos_rot[1, 1]
                                    sep_AU = np.hypot(east_B, north_B)
                                    ab_pa_deg = np.degrees(np.arctan2(east_B, north_B)) % 360
                                    logL = log_likelihood_AB(sep_AU, ab_pa_deg)

                                    writer.writerow({
                                        "r_peri": r_peri, "r_out": r_out, "ecc": ecc,
                                        "peri_direction_deg": peri_dir, "theta_disc_deg": theta,
                                        "phi_disc_deg": phi, "t_after_peri": t_after,
                                        "sep_AU": sep_AU, "ab_pa_deg": ab_pa_deg,
                                        "inc_after": rot["inclination_after"],
                                        "pa_after": rot["pa_after"], "logL": logL})
                                    f.flush()

                                print(f"{count}/{total} physical models | {(time.time()-start)/60:.1f} min", flush=True)

    return output_csv


In [9]:
global_parameter_search(
    R_PERI_VALUES, R_OUT_VALUES, ECC_VALUES, PERI_DIR_VALUES,
    THETA_VALUES, PHI_VALUES, TIME_VALUES,
    n_disc=N_DISC, seed=SEED, output_csv=OUTPUT_CSV
)


1/729 physical models | 0.3 min


2/729 physical models | 0.5 min


3/729 physical models | 0.7 min


4/729 physical models | 1.0 min


5/729 physical models | 1.2 min


...

726/729 physical models | 134.5 min


727/729 physical models | 134.7 min


728/729 physical models | 134.9 min


729/729 physical models | 135.0 min


'rw_aur_global_search.csv'

In [10]:
# Read results and print best 10
rows = []
with open(OUTPUT_CSV, newline="") as f:
    for row in csv.DictReader(f):
        for k in ["r_peri","r_out","ecc","peri_direction_deg","theta_disc_deg","phi_disc_deg",
                  "t_after_peri","sep_AU","ab_pa_deg","inc_after","pa_after","logL"]:
            row[k] = float(row[k])
        rows.append(row)

top10 = sorted(rows, key=lambda r: r["logL"], reverse=True)[:10]

for r in top10:
    print(
        f"logL={r['logL']:.4f} | Rp={r['r_peri']:.0f}, Rout={r['r_out']:.0f}, "
        f"e={r['ecc']:.2f}, peri={r['peri_direction_deg']:.0f}°, "
        f"theta={r['theta_disc_deg']:.0f}°, phi={r['phi_disc_deg']:.0f}°, "
        f"t={r['t_after_peri']:.0f} yr | AB={r['sep_AU']:.1f} AU, PA={r['ab_pa_deg']:.2f}°"
    )


logL=-0.0026 | Rp=55, Rout=80, e=1.01, peri=200°, theta=10°, phi=350°, t=400 yr | AB=208.9 AU, PA=253.92°
logL=-0.0058 | Rp=75, Rout=80, e=1.01, peri=200°, theta=15°, phi=330°, t=500 yr | AB=208.6 AU, PA=253.74°
logL=-0.0061 | Rp=75, Rout=60, e=1.01, peri=200°, theta=15°, phi=330°, t=500 yr | AB=208.6 AU, PA=253.72°
logL=-0.0109 | Rp=55, Rout=80, e=0.95, peri=180°, theta=15°, phi=330°, t=600 yr | AB=208.1 AU, PA=254.72°
logL=-0.0116 | Rp=55, Rout=80, e=0.95, peri=200°, theta=10°, phi=370°, t=400 yr | AB=208.3 AU, PA=253.51°
logL=-0.0137 | Rp=55, Rout=60, e=1.01, peri=200°, theta=10°, phi=350°, t=400 yr | AB=212.0 AU, PA=254.81°
logL=-0.0141 | Rp=55, Rout=70, e=1.01, peri=200°, theta=10°, phi=370°, t=400 yr | AB=207.9 AU, PA=254.80°
logL=-0.0165 | Rp=75, Rout=70, e=1.01, peri=200°, theta=15°, phi=330°, t=500 yr | AB=208.7 AU, PA=253.29°
logL=-0.0180 | Rp=55, Rout=80, e=1.01, peri=200°, theta=10°, phi=330°, t=400 yr | AB=211.5 AU, PA=253.26°
logL=-0.0201 | Rp=55, Rout=60, e=0.95, peri=20